In [311]:
import numpy as np

class BiMatrix:
    def __init__(self, rows, columns, game_type = None):
        """
        Инициализация биматричной игры.

        :param rows: количество строк (стратегий игрока 1)
        :param columns: количество столбцов (стратегий игрока 2)
        :param game_type: тип игры (ordinar, crossroad, prisoner, family)
        """
        self.row = rows
        self.col = columns

        # Генерация случайной игры или задание фиксированных матриц
        if game_type == "ordinar":
            self.matrix_A = np.random.randint(-99, 99, size=(rows, columns))
            self.matrix_B = np.random.randint(-99, 99, size=(rows, columns))

        elif game_type == "crossroad":
            self.matrix_A = np.array([[1, 0.5], [2, 0]])
            self.matrix_B = np.array([[1,   2], [0.5, 0]])

        elif game_type == "prisoner":
            self.matrix_A = np.array([[-5,   0], [-10, -1]])
            self.matrix_B = np.array([[-5, -10], [0,   -1]])

        elif game_type == "family":
            self.matrix_A = np.array([[4, 0], [0, 1]])
            self.matrix_B = np.array([[1, 0], [0, 4]])

    def my_bimatrix(self, a, b):
        # Задаём биматричную игру
        self.matrix_A = a
        self.matrix_B = b        
        
    def nash(self, a, b):
        """
        Находит ситуации равновесия по Нэшу.

        :param a: стратегии первого игрока
        :param b: стратегии второго игрока
        """
        rows, cols = a.shape # Размерности матриц
        nash_equilibria = [] # Список для хранения равновесий по Нэшу
        
        # Перебираем все возможные стратегии
        for i in range(rows):
            for j in range(cols):
                # Выигрыш игроков
                payoff_a = a[i,j]
                payoff_b = b[i,j]

                # Проверяем, является ли (i,j) наилучшим ответом игрока А
                best_response_a = all(payoff_a >= a[ii,j] for ii in range(rows))

                # Проверяем, является ли (i,j) наилучшим ответом игрока B
                best_response_b = all(payoff_b >= b[i,jj] for jj in range(cols))

                # Если оба игрока не хотят менять стратегию -> равновесие по Нэшу
                if best_response_a and best_response_b:
                    nash_equilibria.append((i,j))
        # Выводим результат
        if nash_equilibria:
            print("Ситуации равновесия по Нэшу:")
            for eq in nash_equilibria:
                print(f"({a[eq]:.1f}, {b[eq]:.1f})")
        else:
            print("Равновесие по Нэшу отсутствует")

    def pareto(self, a, b):
        """
        Находит Парето-оптимальные ситуации.

        :param a: стратегии первого игрока
        :param b: стратегии второго игрока
        """
        rows, cols = a.shape
        all_pairs = [(float(a[i, j]), float(b[i, j])) for i in range(rows) for j in range(cols)]

        pareto_optimal = [] # Список для хранения Парето-оптимальных стратегий
        for (ai, bi) in all_pairs:
            is_pareto_optimal = True
            for (ak, bk) in all_pairs:
                if (ak >= ai and bk > bi) or (ak > ai and bk >= bi):  # Улучшение по обоим критериям
                    is_pareto_optimal = False
                    break
            if is_pareto_optimal:
                pareto_optimal.append((ai, bi))

        print(f"Оптимальные по Парето ситуации: {pareto_optimal:}")

    def mixed_nash(self, a, b):
        """
        Вполне смешанная ситуация равновесия по Нэшу

        :param a: стратегии первого игрока
        :param b: стратегии второго игрока
        """
        try:
            A_inv = np.linalg.inv(a)  # Обратная матрица A
            B_inv = np.linalg.inv(b)  # Обратная матрица B
        except np.linalg.LinAlgError:
            print("Матрицы не являются невырожденными, смешанное равновесие не определено")
            return

        u = np.ones((self.row, 1))  # Вектор из единиц

        # Вычисляем коэффициенты v1 и v2
        v1 = 1 / (u.T @ A_inv @ u)
        v2 = 1 / (u.T @ B_inv @ u)

        # Вычисляем стратегии x и y для вполне смешанной ситуации равновесия
        x = (v2 * u.T @ B_inv).flatten() # flatten() -> одномерный массив
        y = (v1 * A_inv @ u).flatten()

        print("Смешанное равновесие по Нэшу:")
        print(f"Вероятности первого игрока: {np.array2string(np.round(x, 3), precision=3, floatmode='fixed')}")
        print(f"Вероятности второго игрока: {np.array2string(np.round(y, 3), precision=3, floatmode='fixed')}")
        print(f"Равновесные выигрыши: v1 = {np.round(v1,3).item():.3f}, v2 = {np.round(v2, 3).item():.3f}")

    def generate_game(self):
        """
        Вывод информации об игре: платежная матрица, равновесие по Нэшу, 
        оптимальные по Парето стратегии, смешанное равновесие.
        """
        a, b = self.matrix_A, self.matrix_B
        self.bimatrix_output(a, b)
        print("\n")
        self.nash(a, b)
        print("\n")
        self.pareto(a, b)
        print("\n")
        self.mixed_nash(a, b)
        print("\n")

    def bimatrix_output(self, a, b):
        """
        Выводит на экран биматричную игру.

        :param a: матрица выплат первого игрока
        :param b: матрица выплат второго игрока
        """
        print('Биматричная игра:')
        for i in range(self.row):
            for j in range(self.col):
                print(f"({a[i, j]:.1f}, {b[i, j]:.1f})", end=' ')
            print()




In [312]:
# game = BiMatrix(2, 2, "prisoner")
# game.generate_game()

# game = BiMatrix(2, 2, "family")
# game.generate_game()

# game = BiMatrix(2, 2, "crossroad")
# game.generate_game()

# obj = BiMatrix(10, 10, "ordinar")
# obj.generate_game()

obj = BiMatrix(2, 2)
a = np.array([[9, 7], [2, 10]])
b = np.array([[8, 4], [1,  3]])
obj.my_bimatrix(a, b)
obj.generate_game()

Биматричная игра:
(9.0, 8.0) (7.0, 4.0) 
(2.0, 1.0) (10.0, 3.0) 


Ситуации равновесия по Нэшу:
(9.0, 8.0)
(10.0, 3.0)


Оптимальные по Парето ситуации: [(9.0, 8.0), (10.0, 3.0)]


Смешанное равновесие по Нэшу:
Вероятности первого игрока: [0.333 0.667]
Вероятности второго игрока: [0.300 0.700]
Равновесные выигрыши: v1 = 7.600, v2 = 3.333


